# 02 - Download OpenStreetMap Data

Downloads OSM `.pbf` files from Geofabrik to the cluster's local disk.
The downloaded files are used by the Nominatim import in notebook 03.

Specify one or more regions as a comma-separated list in the widget
(e.g. `virginia,maryland`) or provide custom URLs.

## Configuration

In [ ]:
dbutils.widgets.text("regions", "monaco", "Regions (comma-separated)")
dbutils.widgets.text("custom_urls", "", "Custom URLs (comma-separated, optional)")
dbutils.widgets.text("output_dir", "/Volumes/justinm/nominatim/osm_data", "Output directory on driver")

In [ ]:
%run ./OSM_SOURCES

In [ ]:
import subprocess
from pathlib import Path

regions_raw = dbutils.widgets.get("regions").strip()
custom_urls_raw = dbutils.widgets.get("custom_urls").strip()
output_dir = Path(dbutils.widgets.get("output_dir"))

regions = [r.strip() for r in regions_raw.split(",") if r.strip()] if regions_raw else []
custom_urls = [u.strip() for u in custom_urls_raw.split(",") if u.strip()] if custom_urls_raw else []

# Validate
for r in regions:
    if r not in OSM_SOURCES:
        raise ValueError(
            f"Unknown region '{r}'. Available: {', '.join(sorted(OSM_SOURCES.keys()))}"
        )

if not regions and not custom_urls:
    raise ValueError("Specify at least one region or custom URL")

# Build download list: (label, url)
downloads = [(r, OSM_SOURCES[r]) for r in regions]
downloads += [(u.split("/")[-1], u) for u in custom_urls]

print("=" * 60)
print("OSM Data Download")
print("=" * 60)
print(f"Output directory: {output_dir}")
print(f"Downloads: {len(downloads)}")
for label, url in downloads:
    print(f"  - {label}: {url}")
print()

## Download Files

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)

downloaded_files = []
for i, (label, url) in enumerate(downloads, 1):
    filename = url.split("/")[-1]
    output_path = output_dir / filename

    print(f"--- [{i}/{len(downloads)}] {label} ---")

    if output_path.exists():
        print(f"  File already exists: {output_path} - reusing")
        downloaded_files.append(str(output_path))
        continue

    print(f"  Downloading from: {url}")
    print(f"  Saving to: {output_path}")
    subprocess.run(
        ["curl", "-L", "-o", str(output_path), url],
        check=True,
    )
    print(f"  Done.")
    downloaded_files.append(str(output_path))
    print()

## Summary

In [ ]:
print("=" * 60)
print("All downloads completed!")
print("=" * 60)
print()
for f in downloaded_files:
    size_mb = Path(f).stat().st_size / (1024 * 1024)
    print(f"  {f}  ({size_mb:.1f} MB)")

print()
print("Next: Run 03_build_nominatim_server with these files.")
print(f"  Regions widget value for notebook 03: {regions_raw}")